# Chop Shop Script Playground
Welcome to the Chop Shop Script Playground! This is a space where you can experiment with different scripts and see how they work in real-time. Whether you're a beginner or an experienced scripter, this playground is designed to work against OWASP BWA for exploit demonstration purposes. Be sure to install the static target the scripts are based off of, to ensure that the scripts run correctly. 



<p>I'll try to make them dynamic but theres no interactive ptty here my dudes</p>


In [1]:
# papermill parameters — override per run:  papermill chopshop.ipynb out.ipynb -p TARGET 10.x.x.x -p EVERY 40
# (in-kernel runs just execute this cell first, or Run All)
TARGET = '192.168.56.107'
EVERY  = 40

In [2]:
# step cell — returns immediately, kernel stays free; TARGET/EVERY come from params
TARGET='192.168.56.107'
async def recon(target=TARGET, every=40):
    job = await nmap(target=target)
    jid = pluck(job, 'job_id')[0]
    status = pluck(job, 'status')[0]
    while status != 'done':
        await asyncio.sleep(every)
        job = await nmap_status(job_id=jid)
        status = pluck(job, 'status')[0]
    return job

recon_task = asyncio.create_task(recon())


2026-10-13 02:54:03,137 [INFO] [TOOL_EXECUTE] Executing Tool ID: auxiliaries.nmap.run_nmap | Path: auxiliaries.nmap.run_nmap | Args: {'target': '192.168.56.107'} | Session: 0


  [repl] Running auxiliaries.nmap.run_nmap
  [repl] Transport: brain_dispatch
  [repl] Args: {"target": "192.168.56.107"}
  [repl] Implementation: auxiliaries.nmap.run_nmap
  [repl] Running auxiliaries.nmap.nmap_status
  [repl] Transport: brain_dispatch
  [repl] Args: {"job_id": "34bc3335"}
  [repl] Implementation: auxiliaries.nmap.nmap_status


In [3]:
job = await recon_task
ports = pluck(job, 'open_ports')[0]
print(ports)

['22/tcp/open/ssh', '80/tcp/open/http', '139/tcp/open/netbios-ssn', '143/tcp/open/imap', '443/tcp/open/ssl/http', '445/tcp/open/netbios-ssn', '5001/tcp/open/java-object', '8080/tcp/open/http', '8081/tcp/open/http']


2026-10-13 02:54:43,278 [INFO] [TOOL_EXECUTE] Executing Tool ID: auxiliaries.nmap.nmap_status | Path: auxiliaries.nmap.nmap_status | Args: {'job_id': '34bc3335'} | Session: 0


# Getting Started? 

i'll be honest chief i've never fuzzed a vulnfeast style box just to test tooling. Also don't really know the OWASP BWA routes well

## Fear not!

We've already gathered a few ports, and each one's got a story 

### Part 1 - The Web 

In [4]:
async def zap_display(target, port, scheme='http'):
    # Construct the full URL
    url = f"{scheme}://{target}:{port}/"
    
    # Call ZAP API
    pagecheck = await zap_open_url(target=url)

    print("pagecheck type:", type(pagecheck))

    # Handle dictionary response - extract HTML content from ZAP's accessUrl structure
    if isinstance(pagecheck, dict):
        html_content = None
        
        # ZAP returns: {'status': 'Success', 'result': {'accessUrl': [{'responseBody': '...html...'}]}, ...}
        result = pagecheck.get('result')
        if isinstance(result, dict):
            access_urls = result.get('accessUrl')
            if isinstance(access_urls, list) and len(access_urls) > 0:
                first_access = access_urls[0]
                if isinstance(first_access, dict):
                    html_content = first_access.get('responseBody')
        
        if html_content is not None:
            pagecheck = html_content
        else:
            print("Could not find HTML content in response dictionary")
            print("Available keys:", list(pagecheck.keys()))
            if 'result' in pagecheck:
                print("Result keys:", list(pagecheck['result'].keys()) if isinstance(pagecheck['result'], dict) else 'not a dict')
            # Changed 'return' to 'continue' or let it fall through to handle the None/Dict case
            # Returning None prevents the function from trying to display later
            return None
        
        # If we successfully extracted HTML, pagecheck is now a string

    # Strip surrounding quotes if present (for JSON strings etc.)
    if isinstance(pagecheck, str):
        if len(pagecheck) >= 2:
            if (pagecheck.startswith('"') and pagecheck.endswith('"')) or \
               (pagecheck.startswith("'") and pagecheck.endswith("'")):
                pagecheck = pagecheck[1:-1]

    # Final validation and display as HTML (not Javascript)
    if not isinstance(pagecheck, str):
        print(f"Expected string for HTML, got {type(pagecheck)}")
        return
        
    display(HTML(pagecheck))

def _scheme(svc):
    # service name straight from nmap's port string; only web services pass
    if 'http' not in svc: return None
    return 'https' if svc.startswith(('ssl', 'https')) else 'http'

# Main loop: route by service name — non-web ports skipped, 443 gets https
async def main():
    displayed, skipped = [], []
    for entry in ports:
        parts = entry.split('/')            # '443/tcp/open/ssl/http'
        port, state, svc = parts[0], parts[2], ('/'.join(parts[3:]) if len(parts) > 3 else '?')
        scheme = _scheme(svc)
        if state != 'open' or scheme is None:
            skipped.append(f"{port}/{svc}")
            continue
        displayed.append(f"{port} ({scheme})")
        await zap_display(target=TARGET, port=port, scheme=scheme)
    print("displayed :", displayed)
    print("skipped   :", skipped)

await main()


  [repl] Running auxiliaries.zap.zap_open_url
  [repl] Transport: brain_dispatch
  [repl] Args: {"target": "http://192.168.56.107:80/"}
  [repl] Implementation: auxiliaries.zap.zap_open_url


2026-10-13 02:54:43,312 [INFO] [TOOL_EXECUTE] Executing Tool ID: auxiliaries.zap.zap_open_url | Path: auxiliaries.zap.zap_open_url | Args: {'target': 'http://192.168.56.107:80/'} | Session: 0


pagecheck type: <class 'dict'>


  [repl] Running auxiliaries.zap.zap_open_url
  [repl] Transport: brain_dispatch
  [repl] Args: {"target": "https://192.168.56.107:443/"}
  [repl] Implementation: auxiliaries.zap.zap_open_url


2026-10-13 02:54:43,433 [INFO] [TOOL_EXECUTE] Executing Tool ID: auxiliaries.zap.zap_open_url | Path: auxiliaries.zap.zap_open_url | Args: {'target': 'https://192.168.56.107:443/'} | Session: 0


...

2026-10-13 02:54:43,606 [INFO] [TOOL_EXECUTE] Executing Tool ID: auxiliaries.zap.zap_open_url | Path: auxiliaries.zap.zap_open_url | Args: {'target': 'http://192.168.56.107:8080/'} | Session: 0


pagecheck type: <class 'dict'>


  [repl] Running auxiliaries.zap.zap_open_url
  [repl] Transport: brain_dispatch
  [repl] Args: {"target": "http://192.168.56.107:8081/"}
  [repl] Implementation: auxiliaries.zap.zap_open_url
pagecheck type: <class 'dict'>


2026-10-13 02:54:43,641 [INFO] [TOOL_EXECUTE] Executing Tool ID: auxiliaries.zap.zap_open_url | Path: auxiliaries.zap.zap_open_url | Args: {'target': 'http://192.168.56.107:8081/'} | Session: 0


displayed : ['80 (http)', '443 (https)', '8080 (http)', '8081 (http)']
skipped   : ['22/ssh', '139/netbios-ssn', '143/imap', '445/netbios-ssn', '5001/java-object']


# Web Enumeration

So we were able to render the pages sequentially via regex and zap_open_url. Awesome! And it looks like we have a few interesting endpoints to explore.

## BWA Apps
### Realistic demonstrations:
 - Webgoat
 - Mutillidae II
 - DVWA
 - bWAPP
 - MCIR
### Old Versions 
 - WordPress
 - OrangeHRM
 - Yazd
 - Joomla
 - AWStats
 - WebCalendar
 - GetBoo
### Applications for Testing:
 - OWASP ZAP-WAVE
 - WAVSEP
 - WIVET